In [1]:
import pandas as pd
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM,
    T5ForConditionalGeneration, 
    T5Tokenizer,
    PreTrainedTokenizer,
    PreTrainedModel
)
import torch
from typing import List, Tuple, Any, Dict, Optional
import warnings
import re
import gc # For garbage collection to free up VRAM

# Suppress warnings
warnings.filterwarnings('ignore')

# --- Model-Specific Configuration ---

# Map NLLB language codes to MADLAD-style <2xx> codes
NLLB_TO_MADLAD_CODE: Dict[str, str] = {
    "eng_Latn": "<2en>",
    "som_Latn": "<2so>",
    # "fra_Latn": "<2fr>",
    # "ara_Arab": "<2ar>",
}

# --- Data Loading Function ---

def load_transcription_data(file_path: str) -> Optional[pd.DataFrame]:
    """
    Loads transcription data from a CSV file into a pandas DataFrame.
    """
    try:
        df = pd.read_csv(file_path)
        print("File loaded successfully!")
        return df
    except FileNotFoundError:
        print(f"Error: The file was not found at the path: {file_path}")
        return None

# --- Model Loading Function ---

def load_model(model_name: str, model_type: str) -> tuple[Any, Any, Any]:
    """
    Load a translation model and tokenizer based on its type.
    """
    print(f"Loading {model_name} (type: {model_type})...")
    
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    
    if model_type == "nllb":
        device_str = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name,
            torch_dtype=dtype
        ).to(device_str)
        device = device_str
        print(f"Model loaded on device: {device}")
        
    elif model_type == "madlad":
        tokenizer = T5Tokenizer.from_pretrained(model_name)
        model = T5ForConditionalGeneration.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=dtype
        )
        device = model.device
        print(f"Model loaded with device_map='auto'. Main device: {device}")
    
    else:
        raise ValueError(f"Unknown model_type: {model_type}. Use 'nllb' or 'madlad'.")

    model.eval()
    return tokenizer, model, device

# --- Text Processing Functions ---

def split_into_sentences(text: str) -> List[str]:
    """
    Improved sentence splitting for Somali text.
    """
    sentences = re.split(r'\s+(?=waxaa|waxa|marka|haddii|sida|taasi)', text, flags=re.IGNORECASE)
    if len(sentences) > 5:
        return [s.strip() for s in sentences if s.strip()]
    
    sentences = re.split(r'\.\s+', text)
    if len(sentences) > 3:
        return [s.strip() for s in sentences if s.strip()]
    
    sentences = re.split(r'\s+(ayaa|oo|iyo|ee)\s+', text, flags=re.IGNORECASE)
    result = []
    for i in range(0, len(sentences), 2):
        if i + 1 < len(sentences):
            result.append(sentences[i] + ' ' + sentences[i + 1])
        else:
            result.append(sentences[i])
    if len(result) > 2:
        return [s.strip() for s in result if s.strip()]
    
    words = text.split()
    chunk_size = 50
    sentences = [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]
    return [s.strip() for s in sentences if s.strip()]

def create_semantic_chunks(
    text: str,
    tokenizer: Any,
    max_tokens: int = 450,
    overlap_sentences: int = 2
) -> List[Tuple[str, int, int]]:
    """
    Split text into chunks that respect 512 token limit.
    """
    sentences = split_into_sentences(text)
    if not sentences:
        return [(text, 0, len(text))]
    
    print(f"  -> Found {len(sentences)} sentences/segments")
    
    chunks = []
    current_chunk = []
    current_tokens = 0
    i = 0
    
    while i < len(sentences):
        sentence = sentences[i]
        sentence_tokens = len(tokenizer.encode(sentence, add_special_tokens=True))
        
        if sentence_tokens > max_tokens:
            print(f"  -> Sentence {i+1} has {sentence_tokens} tokens, splitting by words...")
            words = sentence.split()
            word_chunk = []
            word_tokens = 0
            for word in words:
                word_token_count = len(tokenizer.encode(word + ' ', add_special_tokens=False))
                if word_tokens + word_token_count > max_tokens - 10:
                    if word_chunk:
                        chunks.append((' '.join(word_chunk), 0, 0))
                    word_chunk = [word]
                    word_tokens = word_token_count
                else:
                    word_chunk.append(word)
                    word_tokens += word_token_count
            if word_chunk:
                chunks.append((' '.join(word_chunk), 0, 0))
            i += 1
            continue
        
        if current_tokens + sentence_tokens > max_tokens:
            if current_chunk:
                chunks.append((' '.join(current_chunk), 0, 0))
            overlap_start = max(0, len(current_chunk) - overlap_sentences)
            current_chunk = current_chunk[overlap_start:]
            current_tokens = sum(len(tokenizer.encode(s, add_special_tokens=True)) for s in current_chunk)
        
        current_chunk.append(sentence)
        current_tokens += sentence_tokens
        i += 1
    
    if current_chunk:
        chunks.append((' '.join(current_chunk), 0, 0))
    
    return chunks

def deduplicate_overlap(chunks: List[str], overlap_threshold: int = 5) -> str:
    """
    Intelligently merge chunks by detecting and removing overlapping content.
    """
    if not chunks: return ""
    if len(chunks) == 1: return chunks[0]
    
    result = [chunks[0]]
    for i in range(1, len(chunks)):
        prev_chunk = result[-1]
        curr_chunk = chunks[i]
        prev_words = prev_chunk.split()
        curr_words = curr_chunk.split()
        max_overlap = min(len(prev_words), len(curr_words), 15)
        overlap_found = 0
        
        for overlap_size in range(max_overlap, overlap_threshold - 1, -1):
            prev_end = ' '.join(prev_words[-overlap_size:])
            curr_start = ' '.join(curr_words[:overlap_size])
            if prev_end.lower() == curr_start.lower():
                overlap_found = overlap_size
                break
        
        if overlap_found > 0:
            result.append(' '.join(curr_words[overlap_found:]))
        else:
            result.append(curr_chunk)
    
    return ' '.join(result)

# --- Core Translation Functions ---

def translate_text_chunked(
    text: str,
    tokenizer: Any,
    model: Any,
    device: Any,
    model_type: str,
    src_lang: str,
    tgt_lang: str,
    max_tokens: int = 450,
    overlap_sentences: int = 2
) -> str:
    """
    Translate long text with chunking, supporting different model types.
    """
    if not text or not isinstance(text, str):
        print("  -> Warning: Empty or invalid text provided. Returning empty string.")
        return ""
    
    chunks = create_semantic_chunks(
        text=text,
        tokenizer=tokenizer,
        max_tokens=max_tokens,
        overlap_sentences=overlap_sentences
    )
    print(f"  -> Split into {len(chunks)} chunks for translation")
    
    translated_chunks = []
    
    if model_type == "nllb":
        tokenizer.src_lang = src_lang
        tgt_lang_id = tokenizer.convert_tokens_to_ids(tgt_lang)
        
        for idx, (chunk_text, _, _) in enumerate(chunks):
            if not chunk_text.strip():
                print(f"  -> Skipping empty chunk {idx + 1}/{len(chunks)}")
                continue

            inputs = tokenizer(
                chunk_text,
                return_tensors="pt",
                max_length=512,
                truncation=True,
                padding=True
            ).to(device)
            
            with torch.no_grad():
                translated_tokens = model.generate(
                    **inputs,
                    forced_bos_token_id=tgt_lang_id,
                    max_length=512,
                    num_beams=5,
                    length_penalty=1.0,
                    early_stopping=True,
                    no_repeat_ngram_size=3
                )
            
            translation = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
            translated_chunks.append(translation)
            print(f"  -> Translated chunk {idx + 1}/{len(chunks)} ({len(translation)} chars)")

    elif model_type == "madlad":
        try:
            target_prefix = NLLB_TO_MADLAD_CODE[tgt_lang]
        except KeyError:
            print(f"Error: No MADLAD code found for '{tgt_lang}'.")
            return "[Translation Error: Unknown target language]"
        
        for idx, (chunk_text, _, _) in enumerate(chunks):
            if not chunk_text.strip():
                print(f"  -> Skipping empty chunk {idx + 1}/{len(chunks)}")
                continue
            
            prefixed_text = f"{target_prefix} {chunk_text}"
            
            inputs = tokenizer(
                prefixed_text,
                return_tensors="pt",
                max_length=512,
                truncation=True,
                padding=True
            ).to(model.device)
            
            with torch.no_grad():
                translated_tokens = model.generate(
                    **inputs,
                    max_length=512,
                    num_beams=5,
                    early_stopping=True
                )
            
            translation = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
            translated_chunks.append(translation)
            print(f"  -> Translated chunk {idx + 1}/{len(chunks)} ({len(translation)} chars)")
            
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    full_translation = deduplicate_overlap(translated_chunks)
    return full_translation

def batch_translate_dataframe(
    df: pd.DataFrame,
    text_column: str,
    tokenizer: Any,
    model: Any,
    device: Any,
    model_type: str,
    src_lang: str,
    tgt_lang: str,
    new_column_name: str = "transcript_text_english",
    use_chunking: bool = True
) -> pd.DataFrame:
    """
    Translate a text column in DataFrame, supporting different model types.
    """
    if text_column not in df.columns:
        raise KeyError(f"Column '{text_column}' not found in DataFrame")
    
    df_translated = df.copy()
    translations = []
    
    print(f"Translating {len(df_translated)} rows using {model_type}...")
    
    target_prefix = ""
    tgt_lang_id = None
    
    if model_type == "nllb":
        tokenizer.src_lang = src_lang
        tgt_lang_id = tokenizer.convert_tokens_to_ids(tgt_lang)
    elif model_type == "madlad":
        try:
            target_prefix = NLLB_TO_MADLAD_CODE[tgt_lang]
        except KeyError:
            raise ValueError(f"MADLAD code for '{tgt_lang}' not defined.")
    
    for idx, row in df_translated.iterrows():
        text = row[text_column]
        try:
            positional_idx = df_translated.index.get_loc(idx) + 1
            print(f"\nRow {positional_idx}/{len(df_translated)} (Index: {idx})")
            
            if pd.isna(text) or not isinstance(text, str) or not text.strip():
                print("  -> Skipping row: Text is missing or empty.")
                translations.append(None)
                continue

            if use_chunking:
                translation = translate_text_chunked(
                    text=text,
                    tokenizer=tokenizer,
                    model=model,
                    device=device,
                    model_type=model_type,
                    src_lang=src_lang,
                    tgt_lang=tgt_lang
                )
            else:
                with torch.no_grad():
                    if model_type == "nllb":
                        inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(device)
                        translated_tokens = model.generate(
                            **inputs,
                            forced_bos_token_id=tgt_lang_id,
                            max_length=512
                        )
                    elif model_type == "madlad":
                        prefixed_text = f"{target_prefix} {text}"
                        inputs = tokenizer(prefixed_text, return_tensors="pt", max_length=512, truncation=True).to(model.device)
                        translated_tokens = model.generate(
                            **inputs,
                            max_length=512
                        )
                    translation = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
            
            translations.append(translation)
            print(f"✓ Completed row {positional_idx}")
                
        except Exception as e:
            print(f"✗ Error translating row {idx}: {str(e)}")
            translations.append(None)
    
    df_translated[new_column_name] = translations
    return df_translated

# --- Reusable Pipeline Runner (MODIFIED to return result) ---

def run_translation_pipeline(
    df: pd.DataFrame, 
    model_name: str, 
    model_type: str,
    src_lang: str = "som_Latn",
    tgt_lang: str = "eng_Latn",
    num_rows_to_translate: Optional[int] = None
) -> Optional[Tuple[pd.DataFrame, str]]: # <-- MODIFIED RETURN TYPE
    """
    Loads a model, runs analysis, translates, and cleans up memory.
    Returns a tuple of (translated_DataFrame, new_column_name) on success.
    """
    print("\n" + "#"*80)
    print(f"STARTING PIPELINE FOR: {model_name} (Type: {model_type})")
    print("#"*80)
    
    tokenizer, model, device = None, None, None
    test_df_translated = None
    new_col_name = f"translated_{model_type}_{model_name.split('/')[-1]}"
    
    try:
        # 1. Load model
        tokenizer, model, device = load_model(model_name, model_type)

        # 2. Analyze chunking on the first row
        print("\n" + "="*80)
        print(f"CHUNKING ANALYSIS ({model_name})")
        print("="*80)
        sample_text = df.iloc[0]['transcript_text']
        if pd.isna(sample_text):
            print("Sample text in row 0 is NA. Skipping chunk analysis.")
        else:
            print(f"Sample text: {len(str(sample_text))} chars, {len(str(sample_text).split())} words\n")
            chunks = create_semantic_chunks(sample_text, tokenizer, max_tokens=450, overlap_sentences=2)
            if chunks:
                print(f"\nFinal chunk count: {len(chunks)}")
                avg_chunk_chars = sum(len(c[0]) for c in chunks) / len(chunks)
                print(f"Average chunk size: {avg_chunk_chars:.0f} chars")
            else:
                print("No chunks were created for the sample text.")
        print("="*80)

        # 3. Select rows for translation
        if num_rows_to_translate is None:
            print(f"\nPreparing to translate all {len(df)} rows...")
            df_to_translate = df.copy()
        else:
            print(f"\nPreparing to translate first {num_rows_to_translate} rows...")
            df_to_translate = df.head(num_rows_to_translate).copy()
        
        # 4. Run batch translation
        test_df_translated = batch_translate_dataframe(
            df=df_to_translate,
            text_column="transcript_text",
            tokenizer=tokenizer,
            model=model,
            device=device,
            model_type=model_type,
            src_lang=src_lang,
            tgt_lang=tgt_lang,
            new_column_name=new_col_name,
            use_chunking=True
        )

        # 5. Display results (inside the function)
        print("\n" + "="*80)
        print(f"TRANSLATION RESULTS ({model_name})")
        print("="*80)
        if not test_df_translated.empty:
            for idx, row in test_df_translated.iterrows():
                original = str(row['transcript_text']) if pd.notna(row['transcript_text']) else ""
                translated = str(row[new_col_name]) if pd.notna(row[new_col_name]) else ""
                print(f"\n--- Row {idx} ---")
                print(f"Original: {len(original)} chars")
                print(f"Translated: {len(translated)} chars")
                print(f"\nFirst 500 chars (English):\n{translated[:500]}...")
            print("="*80)
        
        # 6. Return the result on success
        return test_df_translated, new_col_name # <-- MODIFIED

    except Exception as e:
        print(f"ERROR: Pipeline failed for {model_name}.")
        print(f"Details: {e}")
        return None # <-- Return None on failure
    
    finally:
        # 7. Clean up memory
        del model
        del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"\nCleaned up model {model_name} from memory (VRAM cleared).")
        print("#"*80 + "\n")


In [5]:
# --- Main Execution (MODIFIED to capture and print) ---

# 1. Define Data Path
file_name = '/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/02_intermediate/transcripts/mustafaa4a_ASR-Somali/transcriptions_database.csv'

# 2. Load Data
transcriptions_df = load_transcription_data(file_name)

# 3. Run Pipelines (if data loaded successfully)
if transcriptions_df is not None and not transcriptions_df.empty:
    print("\nHere's a preview of your data:")
    print(transcriptions_df.head())

    # --- Example 1: NLLB 3.3B (running on 1 row) ---
    print("\n" + "*"*80)
    print("RUNNING NLLB-200-3.3B")
    print("*"*80)
    
    # Capture the result from the function
    nllb_result = run_translation_pipeline(
        df=transcriptions_df,
        model_name="facebook/nllb-200-3.3B",
        model_type="nllb",
        num_rows_to_translate=1  # Translates first row only
    )
    
    # Check if the result is valid and print it
    if nllb_result:
        df_nllb_result, nllb_col_name = nllb_result
        
        print("\n" + "="*80)
        print("FINAL COMPARISON FOR NLLB-200-3.3B")
        print("="*80)
        
        for idx, row in df_nllb_result.iterrows():
            print(f"\n--- ROW {idx} ---")
            
            print("\nORIGINAL (transcript_text):")
            print(row['transcript_text'])
            
            print(f"\nTRANSLATED ({nllb_col_name}):")
            print(row[nllb_col_name])
            
        print("="*80)

else:
    print("\nData loading failed or DataFrame is empty. Exiting script.")

File loaded successfully!

Here's a preview of your data:
                                 id  \
0  be400344882bded3b69613c55f924523   
1  8919804ce6600b7af5526b91eb18406c   
2  0e8501bdbeb8819eeafac6ddc75efa59   
3  ebf1d24a75fa8d90a2bf4605ec33f62a   
4  192b2896ecdc6563ca66d6671e50c16c   

                                                 url                  title  \
0  https://soundcloud.com/radio-ergo/idaacadda-01...  IDAACADDA 01-JAN-2020   
1  https://soundcloud.com/radio-ergo/idaacadda-03...  IDAACADDA 03-JAN-2020   
2  https://soundcloud.com/radio-ergo/idaacadda-04...  IDAACADDA 04-JAN-2020   
3  https://soundcloud.com/radio-ergo/idaacadda-05...  IDAACADDA 05-JAN-2020   
4  https://soundcloud.com/radio-ergo/idaacadda-06...  IDAACADDA 06-JAN-2021   

   date_recorded              date_processed  processing_duration_seconds  \
0       20200102  2025-10-08T10:22:06.380552                    44.178506   
1       20200103  2025-10-08T10:24:19.056455                    41.640142   
2

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded on device: cuda

CHUNKING ANALYSIS (facebook/nllb-200-3.3B)
Sample text: 44796 chars, 6371 words

  -> Found 204 sentences/segments

Final chunk count: 48
Average chunk size: 1366 chars

Preparing to translate first 1 rows...
Translating 1 rows using nllb...

Row 1/1 (Index: 0)
  -> Found 204 sentences/segments
  -> Split into 48 chunks for translation
  -> Translated chunk 1/48 (405 chars)
  -> Translated chunk 2/48 (250 chars)
  -> Translated chunk 3/48 (625 chars)
  -> Translated chunk 4/48 (639 chars)
  -> Translated chunk 5/48 (437 chars)
  -> Translated chunk 6/48 (472 chars)
  -> Translated chunk 7/48 (268 chars)
  -> Translated chunk 8/48 (643 chars)
  -> Translated chunk 9/48 (470 chars)
  -> Translated chunk 10/48 (349 chars)
  -> Translated chunk 11/48 (484 chars)
  -> Translated chunk 12/48 (560 chars)
  -> Translated chunk 13/48 (210 chars)
  -> Translated chunk 14/48 (610 chars)
  -> Translated chunk 15/48 (601 chars)
  -> Translated chunk 16/48 (461 chars)
 

In [ ]:
# 1. Define Data Path
file_name = '/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/02_intermediate/transcripts/mustafaa4a_ASR-Somali/transcriptions_database.csv'

# 2. Load Data
transcriptions_df = load_transcription_data(file_name)

# --- Example 2: MADLAD 3B (running on 1 row) ---

if 'transcriptions_df' in locals() and transcriptions_df is not None and not transcriptions_df.empty:
    print("\n" + "*"*80)
    print("RUNNING MADLAD-400-3B-MT")
    print("*"*80)
    
    # Capture the result from the function
    madlad_3b_result = run_translation_pipeline(
        df=transcriptions_df,
        model_name="google/madlad400-3b-mt",
        model_type="madlad",
        num_rows_to_translate=1  # Translates first row only
    )
    
    # Check if the result is valid and print it
    if madlad_3b_result:
        df_madlad_3b_result, madlad_3b_col_name = madlad_3b_result
        
        print("\n" + "="*80)
        print("FINAL COMPARISON FOR MADLAD-400-3B-MT")
        print("="*80)
        
        for idx, row in df_madlad_3b_result.iterrows():
            print(f"\n--- ROW {idx} ---")
            
            print("\nORIGINAL (transcript_text):")
            print(row['transcript_text'])
            
            print(f"\nTRANSLATED ({madlad_3b_col_name}):")
            print(row[madlad_3b_col_name])
            
        print("="*80)
else:
    print("\n'transcriptions_df' not found or is empty. Please run the data loading cell first.")

File loaded successfully!

********************************************************************************
RUNNING MADLAD-400-3B-MT
********************************************************************************

################################################################################
STARTING PIPELINE FOR: google/madlad400-3b-mt (Type: madlad)
################################################################################
Loading google/madlad400-3b-mt (type: madlad)...
Model loaded with device_map='auto'. Main device: cuda:0

CHUNKING ANALYSIS (google/madlad400-3b-mt)
Sample text: 44796 chars, 6371 words

  -> Found 204 sentences/segments

Final chunk count: 46
Average chunk size: 1387 chars

Preparing to translate first 1 rows...
Translating 1 rows using madlad...

Row 1/1 (Index: 0)
  -> Found 204 sentences/segments
  -> Split into 46 chunks for translation
  -> Translated chunk 1/46 (510 chars)
  -> Translated chunk 2/46 (510 chars)
  -> Translated chunk 3/46 (510 chars

In [9]:
# --- Example 3: MADLAD 10B (running on 1 row) ---

if 'transcriptions_df' in locals() and transcriptions_df is not None and not transcriptions_df.empty:
    print("\n" + "*"*80)
    print("RUNNING MADLAD-400-10B-MT")
    print("*"*80)
    print("WARNING: This is a 10.7B parameter model and requires significant VRAM.")
    
    # Capture the result from the function
    madlad_10b_result = run_translation_pipeline(
        df=transcriptions_df,
        model_name="google/madlad400-10b-mt",
        model_type="madlad",
        num_rows_to_translate=1  # Translates first row only
    )
    
    # Check if the result is valid and print it
    if madlad_10b_result:
        df_madlad_10b_result, madlad_10b_col_name = madlad_10b_result
        
        print("\n" + "="*80)
        print("FINAL COMPARISON FOR MADLAD-400-10B-MT")
        print("="*80)
        
        for idx, row in df_madlad_10b_result.iterrows():
            print(f"\n--- ROW {idx} ---")
            
            print("\nORIGINAL (transcript_text):")
            print(row['transcript_text'])
            
            print(f"\nTRANSLATED ({madlad_10b_col_name}):")
            print(row[madlad_10b_col_name])
            
        print("="*80)
else:
    print("\n'transcriptions_df' not found or is empty. Please run the data loading cell first.")


********************************************************************************
RUNNING MADLAD-400-10B-MT
********************************************************************************

################################################################################
STARTING PIPELINE FOR: google/madlad400-10b-mt (Type: madlad)
################################################################################
Loading google/madlad400-10b-mt (type: madlad)...
ERROR: Pipeline failed for google/madlad400-10b-mt.
Details: 
T5Tokenizer requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.


Cleaned up model google/madlad400-10b-mt from memory (VRAM cleared).
#########################################################################